# Transformação Bronze → Silver: Products

## 🎯 Objetivo
Transformar a tabela `bronze.bronze_products` aplicando limpeza, padronização de tipos e validações de qualidade para gerar a tabela `silver.products`.

## 📊 Características da Tabela
- **SEM SCD Type 2**: Products não tem versionamento
- **SEM endereço**: Apenas dados de produtos
- **Simples**: Tabela de dimensão básica

## 📦 Estrutura da Tabela
- **product_id**: STRING (chave primária)
- **product_category**: STRING (categoria do produto)
- **product_name**: STRING (nome do produto)
- **sales_price**: DOUBLE (preço de venda)
- **EAN13**: LONG (código de barras 13 dígitos)
- **EAN5**: INT (código de barras 5 dígitos)
- **product_unit**: STRING (unidade de medida)

## 🔄 Transformações Planejadas
1. Limpeza e padronização de strings
2. Validação de preços (sales_price > 0)
3. Remoção de duplicados por product_id
4. Padronização de códigos EAN
5. Adição de colunas de auditoria
6. Validações de qualidade

---

In [0]:
# Imports necessários
from pyspark.sql import functions as F
from pyspark.sql import Window
from datetime import datetime
import uuid

# Configuração de variáveis
CATALOG = "retail_dev"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
BRONZE_TABLE = "bronze_products"
SILVER_TABLE = "products"

# Identificador único desta execução para auditoria
pipeline_run_id = str(uuid.uuid4())
processing_timestamp = datetime.now()

print(f"🔧 Configuração concluída")
print(f"📌 Pipeline Run ID: {pipeline_run_id}")
print(f"⏰ Timestamp de Processamento: {processing_timestamp}")
print(f"📥 Origem: {CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}")
print(f"📤 Destino: {CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}")

# 📖 Etapa 1: Leitura da Tabela Bronze

Carregar os dados da camada bronze e realizar uma análise inicial.

In [0]:
# Ler a tabela bronze
bronze_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}")

# Contagem inicial
initial_count = bronze_df.count()

print(f"✅ Tabela bronze carregada com sucesso!")
print(f"📈 Total de registros: {initial_count:,}")
print(f"\n📊 Schema da tabela bronze:")
bronze_df.printSchema()

# Exibir amostra dos dados
print(f"\n🔍 Amostra dos dados (5 primeiras linhas):")
display(bronze_df.limit(5))

# 🔍 Etapa 2: Análise de Qualidade (PRÉ-TRANSFORMAÇÃO)

Auditoria dos dados antes de aplicar transformações:
- Identificar duplicados
- Contar valores nulos
- Analisar preços inválidos
- Verificar códigos EAN

In [0]:
# 🔵 AUDITORIA: Contagem de duplicados por product_id
duplicates_df = bronze_df.groupBy("product_id").count().filter(F.col("count") > 1)
duplicates_count = duplicates_df.count()
total_duplicate_records = duplicates_df.agg(F.sum("count")).collect()[0][0] if duplicates_count > 0 else 0

print(f"🔴 DUPLICADOS ENCONTRADOS:")
if duplicates_count > 0:
    print(f"   - {duplicates_count:,} product_ids com duplicatas")
    print(f"   - {total_duplicate_records:,} registros duplicados no total")
    print(f"   - {initial_count - duplicates_count:,} registros únicos\n")
else:
    print(f"   - ✅ Nenhum duplicado encontrado!")
    print(f"   - {initial_count:,} registros únicos\n")

# 🔵 AUDITORIA: Análise de valores nulos por coluna
print(f"🟡 ANÁLISE DE VALORES NULOS:")
null_counts = bronze_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in bronze_df.columns
])

null_summary = null_counts.collect()[0].asDict()
for col_name, null_count in sorted(null_summary.items(), key=lambda x: x[1], reverse=True):
    null_pct = (null_count / initial_count) * 100
    if null_count > 0:
        print(f"   - {col_name}: {null_count:,} nulls ({null_pct:.2f}%)")

# 🔵 AUDITORIA: Problemas específicos identificados
print(f"\n🔵 PROBLEMAS ESPECÍFICOS:")

# Preços inválidos (negativos ou zero)
invalid_prices = bronze_df.filter(
    (F.col("sales_price").isNull()) | 
    (F.col("sales_price") <= 0)
).count()
print(f"   - Preços inválidos (≤ 0 ou null): {invalid_prices:,} ({(invalid_prices/initial_count)*100:.1f}%)")

# Códigos EAN inválidos
ean13_null = bronze_df.filter(F.col("EAN13").isNull()).count()
ean5_null = bronze_df.filter(F.col("EAN5").isNull()).count()
print(f"   - EAN13 nulos: {ean13_null:,} ({(ean13_null/initial_count)*100:.1f}%)")
print(f"   - EAN5 nulos: {ean5_null:,} ({(ean5_null/initial_count)*100:.1f}%)")

# product_name vazios
name_empty = bronze_df.filter(
    F.col("product_name").isNull() | (F.trim(F.col("product_name")) == "")
).count()
print(f"   - Product_name vazios: {name_empty:,} ({(name_empty/initial_count)*100:.1f}%)")

print(f"\n📊 Exemplos de registros (primeiros 10):")
display(bronze_df.orderBy("product_id").limit(10))

# 🧹 Etapa 3: Limpeza e Padronização de Tipos

Aplicar transformações para:
- Limpar e padronizar strings (TRIM, UPPER)
- Validar sales_price (substituir inválidos por NULL)
- Padronizar códigos EAN
- Limpar product_unit

In [0]:
# Aplicar todas as transformações em uma única operação
transformed_df = bronze_df.select(
    # --- IDENTIFICAÇÃO ---
    # product_id: Limpar espaços e padronizar
    F.trim(F.upper(F.col("product_id"))).alias("product_id"),
    
    # --- CATEGORIZAÇÃO ---
    # product_category: Limpar espaços e padronizar
    F.trim(F.upper(F.col("product_category"))).alias("product_category"),
    
    # product_name: Limpar espaços e padronizar
    F.trim(F.upper(F.col("product_name"))).alias("product_name"),
    
    # --- PREÇO ---
    # sales_price: Validar (> 0), anular se inválido
    F.when(
        (F.col("sales_price").isNotNull()) & (F.col("sales_price") > 0),
        F.col("sales_price")
    ).otherwise(None).alias("sales_price"),
    
    # --- CÓDIGOS EAN ---
    # EAN13: Manter BIGINT (código de barras 13 dígitos)
    F.col("EAN13"),
    
    # EAN5: Manter INT (código de barras 5 dígitos)
    F.col("EAN5"),
    
    # --- UNIDADE ---
    # product_unit: Limpar espaços e padronizar
    F.trim(F.upper(F.col("product_unit"))).alias("product_unit")
)

print("✅ Transformações aplicadas com sucesso!")
print(f"\n📊 Novo schema após transformações:")
transformed_df.printSchema()

print(f"\n🔍 Amostra dos dados transformados (5 primeiras linhas):")
display(transformed_df.limit(5))

# 🗑️ Etapa 4: Remoção de Duplicados

Products **não tem versionamento** (sem SCD Type 2).

Estratégia:
- Remover duplicados por product_id
- Manter o primeiro registro encontrado
- **Garantir** que cada product_id apareça apenas uma vez na Silver

In [0]:
# Estratégia: Remover duplicados mantendo primeiro registro por product_id
window_spec = Window.partitionBy("product_id").orderBy(F.lit(1))

deduplicated_df = transformed_df.withColumn(
    "row_num",
    F.row_number().over(window_spec)
).filter(
    F.col("row_num") == 1
).drop("row_num")

# Contagem após remoção de duplicados
silver_count = deduplicated_df.count()
removed_duplicates = initial_count - silver_count

print("✅ Duplicados removidos com sucesso!")
print(f"\n🔵 AUDITORIA - REMOÇÃO DE DUPLICADOS:")
print(f"   - Registros na Bronze: {initial_count:,}")
print(f"   - Registros na Silver: {silver_count:,}")
print(f"   - Duplicados removidos: {removed_duplicates:,}")
print(f"   - Redução: {(removed_duplicates/initial_count)*100:.2f}%")

# Verificar se product_id é único
unique_products = deduplicated_df.select("product_id").distinct().count()
print(f"\n✅ Verificação de unicidade:")
print(f"   - Total de registros: {silver_count:,}")
print(f"   - Product_ids únicos: {unique_products:,}")
if silver_count == unique_products:
    print(f"   - ✅ SUCESSO: Cada product_id é único na Silver!")
else:
    print(f"   - ⚠️ ALERTA: Ainda existem duplicados!")

# ✅ Etapa 5: Validações Finais (PÓS-TRANSFORMAÇÃO)

Verificar a qualidade dos dados transformados:
- Confirmar unicidade de product_id
- Analisar valores nulos após limpeza
- Validar preços
- Verificar códigos EAN

In [0]:
print("🔵 AUDITORIA - VALIDAÇÕES PÓS-TRANSFORMAÇÃO")
print("="*60)

# 1. Confirmar unicidade de product_id
print(f"\n1️⃣ UNICIDADE DE PRODUCT_ID:")
duplicates_check = deduplicated_df.groupBy("product_id").count().filter(F.col("count") > 1).count()
if duplicates_check == 0:
    print(f"   ✅ Product_id é único (0 duplicados)")
else:
    print(f"   ⚠️ {duplicates_check} product_ids ainda estão duplicados!")

# 2. Análise de valores nulos após limpeza
print(f"\n2️⃣ VALORES NULOS APÓS LIMPEZA:")
null_counts_after = deduplicated_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in deduplicated_df.columns
])

null_summary_after = null_counts_after.collect()[0].asDict()
for col_name, null_count in sorted(null_summary_after.items(), key=lambda x: x[1], reverse=True):
    null_pct = (null_count / silver_count) * 100
    if null_count > 0:
        print(f"   - {col_name}: {null_count:,} nulls ({null_pct:.2f}%)")

# 3. Validar preços
print(f"\n3️⃣ VALIDAÇÃO DE PREÇOS:")
valid_prices = deduplicated_df.filter(F.col("sales_price").isNotNull()).count()
invalid_prices_after = deduplicated_df.filter(F.col("sales_price").isNull()).count()
if valid_prices > 0:
    min_price = deduplicated_df.filter(F.col("sales_price").isNotNull()).agg(F.min("sales_price")).collect()[0][0]
    max_price = deduplicated_df.agg(F.max("sales_price")).collect()[0][0]
    avg_price = deduplicated_df.agg(F.avg("sales_price")).collect()[0][0]
    print(f"   - Preços válidos: {valid_prices:,} ({(valid_prices/silver_count)*100:.1f}%)")
    print(f"   - Preços inválidos: {invalid_prices_after:,} ({(invalid_prices_after/silver_count)*100:.1f}%)")
    print(f"   - Preço mínimo: ${min_price:.2f}")
    print(f"   - Preço máximo: ${max_price:.2f}")
    print(f"   - Preço médio: ${avg_price:.2f}")

# 4. Análise de códigos EAN
print(f"\n4️⃣ ANÁLISE DE CÓDIGOS EAN:")
ean13_filled = deduplicated_df.filter(F.col("EAN13").isNotNull()).count()
ean5_filled = deduplicated_df.filter(F.col("EAN5").isNotNull()).count()
print(f"   - EAN13 preenchidos: {ean13_filled:,} ({(ean13_filled/silver_count)*100:.1f}%)")
print(f"   - EAN5 preenchidos: {ean5_filled:,} ({(ean5_filled/silver_count)*100:.1f}%)")

print(f"\n" + "="*60)
print("✅ Validações concluídas!")

# 📋 Etapa 6: Adição de Colunas de Auditoria

Adicionar colunas de rastreabilidade e qualidade:
- **data_quality_score**: Score de qualidade (0-100) baseado em completude
- **processed_at**: Timestamp do processamento
- **source_table**: Tabela de origem (bronze)
- **pipeline_run_id**: Identificador único desta execução

In [0]:
# Calcular data quality score para cada registro
# Score = (número de campos não-nulos / total de campos) * 100

data_columns = [
    "product_id", "product_category", "product_name", "sales_price",
    "EAN13", "EAN5", "product_unit"
]

# Contar campos não-nulos para cada registro
non_null_counts = sum([F.when(F.col(c).isNotNull(), 1).otherwise(0) for c in data_columns])

# Adicionar colunas de auditoria
final_df = deduplicated_df.withColumn(
    "data_quality_score",
    (non_null_counts / len(data_columns) * 100).cast("int")
).withColumn(
    "processed_at",
    F.lit(processing_timestamp)
).withColumn(
    "source_table",
    F.lit(f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}")
).withColumn(
    "pipeline_run_id",
    F.lit(pipeline_run_id)
)

print("✅ Colunas de auditoria adicionadas com sucesso!")
print(f"\n📊 Schema final com auditoria:")
final_df.printSchema()

print(f"\n🔍 Amostra com metadados (3 primeiras linhas):")
display(final_df.limit(3))

# Estatísticas de qualidade
print(f"\n📈 ESTATÍSTICAS DE QUALIDADE:")
quality_stats = final_df.groupBy("data_quality_score").count().orderBy("data_quality_score", ascending=False)
print(f"\nDistribuição do Data Quality Score:")
display(quality_stats)

In [0]:
# 🔤 PADRONIZAÇÃO: Converter TODOS os nomes de colunas para UPPER
print("🔤 Padronizando nomes de colunas para UPPER...")
final_df = final_df.select([F.col(c).alias(c.upper()) for c in final_df.columns])
print("✅ Todas as colunas convertidas para UPPER!")

print(f"\n📊 Colunas finais:")
print(final_df.columns)

print(f"\n📋 Schema final:")
final_df.printSchema()

print(f"\n🔍 Amostra (3 primeiras linhas):")
display(final_df.limit(3))

# 💾 Etapa 7: Escrita na Tabela Silver

Gravar os dados transformados na camada Silver com:
- Formato Delta Lake
- Change Data Feed habilitado
- Otimização e estatísticas

In [0]:
# Nome completo da tabela silver
silver_table_name = f"{CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}"

print(f"💾 Gravando dados na tabela Silver: {silver_table_name}")
print(f"📈 Total de registros a gravar: {final_df.count():,}")

# Gravar tabela Delta com Change Data Feed
final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable(silver_table_name)

print(f"\n✅ Dados gravados com sucesso!")

# Otimizar a tabela
print(f"\n🛠️ Otimizando tabela Delta...")
spark.sql(f"OPTIMIZE {silver_table_name}")
print("✅ OPTIMIZE concluído!")

# Coletar estatísticas
print(f"\n📊 Coletando estatísticas da tabela...")
spark.sql(f"ANALYZE TABLE {silver_table_name} COMPUTE STATISTICS")
print("✅ ANALYZE TABLE concluído!")

# Verificar a tabela criada
print(f"\n🔍 Verificação final da tabela Silver:")
silver_verification = spark.table(silver_table_name)
print(f"   - Total de registros: {silver_verification.count():,}")
print(f"   - Número de colunas: {len(silver_verification.columns)}")
print(f"\n📊 Colunas da tabela Silver:")
print(silver_verification.columns)

print(f"\n✅ Transformação Bronze → Silver concluída com sucesso!")

# 📊 Relatório Final de Transformação

Resumo completo do pipeline Bronze → Silver para Products

In [0]:
# Recuperar dados finais da Silver
silver_final_df = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}")
final_count = silver_final_df.count()

# Recuperar processed_at do DataFrame
processed_at_value = silver_final_df.select("PROCESSED_AT").first()[0]

# Dados para o relatório
summary_data = {
    "pipeline_run_id": str(pipeline_run_id),
    "processed_at": str(processed_at_value),
    "source_table": f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}",
    "destination_table": f"{CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}",
    "initial_records_bronze": str(initial_count),
    "final_records_silver": str(final_count),
    "duplicates_removed": str(removed_duplicates),
    "reduction_percentage": str(round((removed_duplicates/initial_count)*100, 2)) if initial_count > 0 else "0",
    "columns_count": str(len(silver_final_df.columns))
}

print("="*70)
print("📄 RELATÓRIO FINAL - TRANSFORMAÇÃO BRONZE → SILVER (PRODUCTS)")
print("="*70)

print(f"\n🔑 IDENTIFICAÇÃO DA EXECUÇÃO:")
print(f"   Pipeline Run ID: {summary_data['pipeline_run_id']}")
print(f"   Data/Hora Processamento: {summary_data['processed_at']}")

print(f"\n📦 TABELAS:")
print(f"   Origem: {summary_data['source_table']}")
print(f"   Destino: {summary_data['destination_table']}")

print(f"\n📈 ESTATÍSTICAS DE REGISTROS:")
print(f"   Registros na Bronze: {summary_data['initial_records_bronze']}")
print(f"   Registros na Silver: {summary_data['final_records_silver']}")
print(f"   Duplicados Removidos: {summary_data['duplicates_removed']}")
print(f"   Taxa de Redução: {summary_data['reduction_percentage']}%")

print(f"\n📊 ESTRUTURA:")
print(f"   Número de Colunas: {summary_data['columns_count']}")

# Estatísticas de qualidade
print(f"\n🎯 QUALIDADE DOS DADOS:")
quality_distribution = silver_final_df.groupBy(
    (F.floor(F.col("DATA_QUALITY_SCORE") / 10) * 10).alias("quality_bucket")
).count().orderBy("quality_bucket", ascending=False)

print("\nDistribuição por Faixa de Qualidade:")
for row in quality_distribution.collect():
    bucket = int(row['quality_bucket'])
    count = row['count']
    pct = (count / final_count) * 100
    print(f"   {bucket}-{bucket+9}%: {count:,} registros ({pct:.1f}%)")

avg_quality = silver_final_df.agg(F.avg("DATA_QUALITY_SCORE")).collect()[0][0]
min_quality = silver_final_df.agg(F.min("DATA_QUALITY_SCORE")).collect()[0][0]
max_quality = silver_final_df.agg(F.max("DATA_QUALITY_SCORE")).collect()[0][0]

print(f"\nMétricas de Qualidade:")
print(f"   Quality Score Médio: {avg_quality:.1f}%")
print(f"   Quality Score Mínimo: {min_quality}%")
print(f"   Quality Score Máximo: {max_quality}%")

# Análise de preços
print(f"\n💰 ANÁLISE DE PREÇOS:")
valid_prices_final = silver_final_df.filter(F.col("SALES_PRICE").isNotNull()).count()
if valid_prices_final > 0:
    min_price_final = silver_final_df.filter(F.col("SALES_PRICE").isNotNull()).agg(F.min("SALES_PRICE")).collect()[0][0]
    max_price_final = silver_final_df.agg(F.max("SALES_PRICE")).collect()[0][0]
    avg_price_final = silver_final_df.agg(F.avg("SALES_PRICE")).collect()[0][0]
    print(f"   Produtos com preço: {valid_prices_final:,} ({(valid_prices_final/final_count)*100:.1f}%)")
    print(f"   Preço Mínimo: ${min_price_final:.2f}")
    print(f"   Preço Máximo: ${max_price_final:.2f}")
    print(f"   Preço Médio: ${avg_price_final:.2f}")

print(f"\n" + "="*70)
print("✅ PIPELINE BRONZE → SILVER CONCLUÍDO COM SUCESSO!")
print("="*70)

print(f"\n📊 Visualização dos dados finais (5 primeiras linhas):")
display(silver_final_df.orderBy("PRODUCT_ID").limit(5))